# 13b — Denominator ruling (study plan v0.15): F on the 12 DESIGN formulations as PRIMARY (zero solves)

s1x and s3x (S1/S3 shares under the θ=3× carbon regime, SSP585 only) are reclassified from voting to **DIAGNOSTIC**:
not defensible value positions, deep-carbon regime over-represented (4/14 votes), climate asymmetry (8:6), and a
failed-target state carried into F. Handling: the manifest gains `role` (as a NEW version — `spec/manifest_v2.csv`
+ sha; the frozen v1 file and its hash are untouched), and F / bands / E1 / E11 are recomputed on the 12 design
cells as PRIMARY, with the 14-cell as-frozen results (13) kept for the supplement with per-cell deltas.
Runs after 13 (reads its E11 matrices); writes nothing 13 wrote. Kernel `y2y-geo`, ~5 min (streams 14 pools).

In [ ]:
# ---- bootstrap + manifest v2 (role) + stream the 14 unguarded surfaces --------------------------------
import importlib, json, hashlib, pathlib, sys
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]; sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec
for _m in (config, lc, ec):
    importlib.reload(_m)
SPEC = ROOT / "analyses" / "y2y" / "spec"; RUNS = ROOT / "analyses" / "y2y" / "runs"; FIG = ROOT / "analyses" / "y2y" / "figures"
MAN = pd.read_csv(SPEC / "manifest.csv"); assert len(MAN) == 14
digest = hashlib.sha256((SPEC / "manifest.csv").read_bytes()).hexdigest()
frozen = (SPEC / "manifest_freeze.sha256").read_text().split()[0]
assert digest == frozen, "spec/manifest.csv differs from the frozen pre-registration hash -- stop"
DIAG = ("s1x", "s3x")
M2 = MAN.copy(); M2["role"] = np.where(M2.scenario_id.isin(DIAG), "diagnostic", "design")
M2.to_csv(SPEC / "manifest_v2.csv", index=False)
d2 = hashlib.sha256((SPEC / "manifest_v2.csv").read_bytes()).hexdigest()
(SPEC / "manifest_v2.sha256").write_text(f"{d2}  manifest_v2.csv  (= frozen v1 {frozen[:16]}... + role column; study plan v0.15, 2026-09-08)\n")
DESIGN = list(M2[M2.role == "design"].formulation_id); ALL = list(M2.formulation_id)
print(f"manifest_v2: {len(DESIGN)} design + {len(ALL) - len(DESIGN)} diagnostic ({', '.join(M2[M2.role == 'diagnostic'].formulation_id)}) | sha256 {d2[:16]}...")

pu = lc.pu_mask()
with rasterio.open(config.HANDOFF_DIR / "mask_protected_areas.tif") as src:
    locked = (src.read(1) == 1) & pu
disc = ~locked[pu]
F_FORM, ANCHORS = {}, {}
for fid in ALL:                                   # same artifacts and estimator as 13 cell 1 (UNGUARDED band)
    cd = RUNS / fid
    A = ec.read_selections(cd / "anchor.tif", pu)[0]
    M = ec.read_selections(cd / "mga_g05.tif", pu)
    F_FORM[fid] = np.vstack([A[None, :], M]).mean(axis=0).astype(np.float32); ANCHORS[fid] = A
    del M
    print(f"{fid:<22} f frequent {int((F_FORM[fid][disc] >= 0.70).sum()):>7,} km2 | role {M2.set_index('formulation_id').loc[fid, 'role']}")


In [ ]:
# ---- F12 (design, PRIMARY) vs F14 (as frozen, supplement): per-cell deltas, bands, E1 --------------------
F14 = np.mean([F_FORM[c] for c in ALL], axis=0); F12 = np.mean([F_FORM[c] for c in DESIGN], axis=0)
N14 = np.mean([ANCHORS[c] for c in ALL], axis=0); N12 = np.mean([ANCHORS[c] for c in DESIGN], axis=0)
d = (F12 - F14)[disc]
print(f"per-cell F12 - F14 over unprotected land: mean {d.mean():+.4f} | mean |d| {np.abs(d).mean():.4f} | max |d| {np.abs(d).max():.4f} "
      f"(analytic bound 2/14 = {2/14:.4f}: two votes removed; the single-vote bound 1/14 = {1/14:.4f})")
print(f"cells with |d| > 1/14: {int((np.abs(d) > 1/14).sum()):,} | |d| > 0.05: {int((np.abs(d) > 0.05).sum()):,}")
BANDS = [("always (>=0.95)", 0.95, 1.01), ("frequent (0.70-0.95)", 0.70, 0.95), ("conditional (0.30-0.70)", 0.30, 0.70),
         ("rare (0.05-0.30)", 0.05, 0.30), ("never (<0.05)", -0.01, 0.05)]
rows = []
for name, lo, hi in BANDS:
    m14 = (F14 >= lo) & (F14 < hi) & disc; m12 = (F12 >= lo) & (F12 < hi) & disc
    rows.append(dict(band=name, F14_km2=int(m14.sum()), F12_km2=int(m12.sum()), delta_km2=int(m12.sum()) - int(m14.sum()),
                     jaccard=float((m14 & m12).sum() / max((m14 | m12).sum(), 1))))
T = pd.DataFrame(rows)
print("\nbands (unprotected km2), 14 as frozen vs 12 design:"); print(T.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
core14, core12 = (F14 >= 0.70) & disc, (F12 >= 0.70) & disc
print(f"frequent tier (>= 0.70): {int(core14.sum()):,} -> {int(core12.sum()):,} km2 | Jaccard {float((core14 & core12).sum() / max((core14 | core12).sum(), 1)):.3f}")
b14, b12 = (F14 - N14)[disc], (F12 - N12)[disc]
print(f"E1 bias (hierarchical - naive): 14-cell mean |b| {np.abs(b14).mean():.4f} / max {np.abs(b14).max():.3f} | "
      f"12-design mean |b| {np.abs(b12).mean():.4f} / max {np.abs(b12).max():.3f}")
# products: the PRIMARY surface + the supplement table + a figure
out = RUNS / "ensemble_v1"; out.mkdir(exist_ok=True)
with rasterio.open(config.HANDOFF_DIR / "cost_uniform.tif") as ref:
    prof = ref.profile | dict(dtype="float32", count=1, nodata=np.nan)
Gd = np.full(pu.shape, np.nan, np.float32); Gd[pu] = F12
with rasterio.open(out / "F_surface_design12.tif", "w", **prof) as dst:
    dst.write(Gd, 1)
T["note"] = ""
T.loc[len(T)] = dict(band="per-cell |F12-F14| mean / max", F14_km2=np.nan, F12_km2=np.nan, delta_km2=np.nan, jaccard=np.nan,
                     note=f"{np.abs(d).mean():.4f} / {np.abs(d).max():.4f} (bound 2/14)")
T.loc[len(T)] = dict(band="E1 mean|bias| 14 / 12", F14_km2=np.nan, F12_km2=np.nan, delta_km2=np.nan, jaccard=np.nan,
                     note=f"{np.abs(b14).mean():.4f} / {np.abs(b12).mean():.4f}")
T.to_csv(SPEC / "T_denominator_v015.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(18, 9))
for ax, (name, surf, cm, vmin, vmax) in zip(axes, (("F12 (design, PRIMARY)", F12, "viridis", 0, 1), ("F14 (as frozen)", F14, "viridis", 0, 1),
                                                    ("F12 - F14", F12 - F14, "coolwarm", -2/14, 2/14))):
    Gp = np.full(pu.shape, np.nan, np.float32); Gp[pu] = surf
    im = ax.imshow(Gp, cmap=cm, vmin=vmin, vmax=vmax); ax.set_title(name); ax.axis("off"); fig.colorbar(im, ax=ax, shrink=0.5)
fig.savefig(FIG / "gate4_F_design12_vs_14.png", dpi=180, bbox_inches="tight"); plt.show()
print(f"\nwrote runs/ensemble_v1/F_surface_design12.tif, spec/T_denominator_v015.csv, figures/gate4_F_design12_vs_14.png")


In [ ]:
# ---- E11 recount on the 12 design cells (from 13's saved matrices; zero solves) --------------------------
D = pd.read_csv(SPEC / "E11_delta_matrix.csv", index_col=0); J = pd.read_csv(SPEC / "E11_anchor_jaccard.csv", index_col=0)
def inband(M):
    off = ~np.eye(len(M), dtype=bool); v = M.values.astype(float)
    return int((v[off] <= 0.05 + 1e-9).sum()), int(off.sum())
n14 = inband(D.loc[ALL, ALL]); n12 = inband(D.loc[DESIGN, DESIGN])
print(f"E11 ordered pairs mutually in-band (Delta <= 5%): 14-cell {n14[0]}/{n14[1]} -> 12-design {n12[0]}/{n12[1]}")
fails = [(a, b, float(D.loc[a, b])) for a in ALL for b in ALL if a != b and float(D.loc[a, b]) > 0.05 + 1e-9]
diag_involved = [(a, b, v) for a, b, v in fails if a.startswith(DIAG) or b.startswith(DIAG)]
print(f"14-cell out-of-band pairs: {len(fails)}; involving a diagnostic cell as plan or objective: {len(diag_involved)} "
      f"(as objective: {sum(b.startswith(DIAG) for _, b, _ in diag_involved)}, as plan: {sum(a.startswith(DIAG) for a, _, _ in diag_involved)})")
for a, b, v in sorted(fails, key=lambda r: -r[2]):
    print(f"   plan {a:<22} under objective {b:<22} Delta {v:.3f}{'   <- diagnostic involved' if (a, b, v) in diag_involved else ''}")
offJ = J.loc[DESIGN, DESIGN].values.astype(float)[~np.eye(12, dtype=bool)]
print(f"between-anchor Jaccard, 12 design: min {offJ.min():.3f} / mean {offJ.mean():.3f} / max {offJ.max():.3f}")
rec = dict(e11_pairs_14=n14, e11_pairs_12=n12, out_of_band_14=len(fails), out_of_band_involving_diagnostic=len(diag_involved),
           out_of_band_12=n12[1] - n12[0], design=DESIGN, diagnostic=[f for f in ALL if f.startswith(DIAG)])
(SPEC / "E11_recount_v015.json").write_text(json.dumps(rec, indent=1))
print("wrote spec/E11_recount_v015.json -- results_log R10.16")


## Next
Study plan v0.15 is now on disk: `spec/manifest_v2.csv` (+ sha) carries `role`; the package (19/20) reads it through
`director_core.package_manifest`. E3/E7 keep using the diagnostics (13 unchanged). Log: methods_log M4.23, results_log R10.16.